In [ ]:
import altair as alt
from google.colab import auth
auth.authenticate_user()

In [ ]:
GCP_PROJECT = 'httparchive'  # @param {type: "string"}

In [Part 1](https://colab.research.google.com/github/HTTPArchive/har.fyi/blob/main/workbooks/exploring_httparchive-all-pages_tables.ipynb) we examined the `pages` tables. Then in [Part 2](https://colab.research.google.com/github/HTTPArchive/har.fyi/blob/main/workbooks/exploring_httparchive-all-requests_tables.ipynb) we worked with the `requests` tables. Now let's look at how we can combine both the `requests` and `pages` tables to perform a deeper analysis. In the following example, we're going to investigate the following question:
  "Is image weight more commonly a factor of 1 large image, or many smaller images?".

Let's start with a simple query against the `pages` table. This is a simple query that just selects the URL, the number of images and the bytes of those images.

In [ ]:
# This query will process 25 GB when run.
%%bigquery --project {GCP_PROJECT}
SELECT
  page,
  INT64(summary.reqImg) AS image_requests,
  INT64(summary.bytesImg) AS image_bytes
FROM `httparchive.crawl.pages`
WHERE date = '2024-05-01'
    AND client='desktop'
    AND is_root_page
ORDER BY
  image_bytes DESC

In the previous example we didn't use the requests table at all. However if we wanted to, we could do the same analysis by JOINing the request table. In order to do this, I've made the following modifications to the query above:

* Named the tables (i.e., so we can reference "httparchive.all.pages" as "pages")
* Reference the tables for some of the columns that exist in both tables (such as page)
* Since we're focusing on images, we'll include `requests.type="image"` in our WHERE clause.
* JOIN the tables `ON requests.page = pages.page`

The below query is similar to the previous one, but uses both tables. The unique differentiation here is that `requests` table is much bigger and we are applying sampling to the `requests` table for a faster and cheaper result.

In [ ]:
# This query will process 80 GB when run.
%%bigquery --project {GCP_PROJECT}
WITH pages AS (
  SELECT
    page
  FROM `httparchive.crawl.pages`
  WHERE date = '2024-05-01'
      AND client='desktop'
      AND is_root_page
), requests AS (
  SELECT
    page,
    COUNT(0) AS image_requests,
    SUM(INT64(summary.respBodySize)) AS image_bytes
  FROM `httparchive.crawl.requests` TABLESAMPLE SYSTEM (10 PERCENT)
  WHERE date = '2024-05-01'
      AND client='desktop'
      AND is_root_page
      AND type = "image"
  GROUP BY
    page
)

SELECT
  pages.page,
  image_requests,
  image_bytes
FROM requests
LEFT JOIN pages
ON requests.page = pages.page
ORDER BY
  image_bytes DESC

Some of these image weights seem quite extreme, but these represent the most image heavy sites in the archive. If you want to validate this, you can load one of the pages in DevTools or WebPageTest and see for yourself.  For example based on the results above, https://princetel.com/ has 1 image totaling 110MB. WebPageTest confirms that this is accurate (although the image weight likely changed slightly between when the HTTP Archive saw the page vs when this manual test was performed.

# Content breakdown by mime/type

In [ ]:
# This query will process 131 GB when run.
%%bigquery df_requests_type --project {GCP_PROJECT}
SELECT
    type,
    COUNT(0) AS requests,
    SUM(INT64(summary.respBodySize)) AS bytes
FROM `httparchive.crawl.requests` TABLESAMPLE SYSTEM (5 PERCENT)
WHERE date = '2024-05-01'
    AND client = 'desktop'
    AND is_root_page
    AND page = 'https://princetel.com/'
GROUP BY
    type
ORDER BY
    requests DESC

In [ ]:
df_requests_type.head()

In [ ]:
pie_chart_requests = alt.Chart(df_requests_type).mark_arc().encode(
    theta=alt.Theta(field="requests", type="quantitative", stack=True),
    color=alt.Color(field="type", type="nominal", sort=None),
    order=alt.Order(field="requests", type="quantitative", sort='descending'),
    tooltip=["type", alt.Tooltip("requests:Q", format=".0")]
).properties(
    title="Requests by Type"
)
pie_chart_requests + pie_chart_requests.mark_text(
    radius=160
).encode(
    text=alt.Text("requests:Q", format=".0")
)

In [ ]:
pie_chart_bytes = alt.Chart(df_requests_type).mark_arc().encode(
    theta=alt.Theta(field="bytes", type="quantitative", stack=True),
    color=alt.Color(field="type", type="nominal", sort=None),
    order=alt.Order(field="bytes", type="quantitative", sort='descending'),
    tooltip=["type", alt.Tooltip("bytes:Q", format=".1s")]
).properties(
    title="Bytes by Type"
)
pie_chart_bytes + pie_chart_bytes.mark_text(
    radius=160
).encode(
    text=alt.Text("bytes:Q", format=".1s")
)

If this was all the information we needed, then JOINING the `pages` table would have been unecessary. All the requests data can be aggregated using `page` column in `requests` table.

Another example we will look into is the requests for JS scripts and page `_TotalBlockingTime` metric within the `payload` column. We are using the `page` column to classify whether the `requests` data is first or third party content.

The query below gets the median Total Blocking Time (TBT) per a number of first and third-party scripts.

In [ ]:
# This query will process 47 GB when run.
%%bigquery df_tbt_scripts --project {GCP_PROJECT}
WITH pages AS (
  SELECT
    page,
    CAST(ROUND(FLOAT64(payload._TotalBlockingTime)/100)*100 AS INT64) AS TBT_bucket
  FROM `httparchive.crawl.pages` TABLESAMPLE SYSTEM (1 PERCENT)
  WHERE date = '2024-05-01'
      AND client = 'desktop'
      AND is_root_page
  GROUP BY
    page,
    TBT_bucket
), requests AS (
  SELECT
    page,
    NET.REG_DOMAIN(url) = NET.REG_DOMAIN(page) AS is_1p_script,
    COUNT(0) AS count
  FROM `httparchive.crawl.requests` TABLESAMPLE SYSTEM (10 PERCENT)
  WHERE date = '2024-05-01'
      AND client='desktop'
      AND is_root_page
      AND type = "script"
  GROUP BY
    page,
    is_1p_script
  HAVING is_1p_script IS NOT NULL
)

SELECT
  is_1p_script,
  count,
  APPROX_QUANTILES(TBT_bucket, 100)[50] AS TBT_median
FROM requests
INNER JOIN pages
ON requests.page = pages.page
GROUP BY
  count,
  is_1p_script
ORDER BY
  count,
  is_1p_script

In [ ]:
df_tbt_scripts

In [ ]:
# filter outliers
df_filtered = df_tbt_scripts[(df_tbt_scripts['TBT_median'] < 20000) & (df_tbt_scripts['count'] < 60)]

base = alt.Chart(df_filtered).mark_circle(opacity=0.5).encode(
    alt.X('count:Q').title('Script Count'),
    alt.Y('TBT_median:Q').axis(format='s').title('Total Blocking Time, ms'),
    alt.Color('is_1p_script:N').title('1st Party Script'),
    tooltip=['TBT_median', 'is_1p_script', 'count']
).properties(
    title="Total Blocking Time (ms) by 1st/3rd Script Count",
    width=1200,
    height=600
)

base + base.transform_loess('count', 'TBT_median', groupby=['is_1p_script']).mark_line()

**Note**: as the query in this guide is sampled the results may misrepresent the true metrics values.

General trend is that 3rd-party scripts tend to have a bigger correllation with the TBT than 1st-party. The TBT tends to increase as the number of 3rd-party scripts increases up to 20 scripts of a particular type per page.

After 20 scripts per page the TBT increases more 3rd-party scripts added to a page.

You can also make a copy of the workbook and experiment with some of your own visualization ideas for the data as well.